# Кредитный скоринг физических лиц
Учебный проект LendingClub. **Ограничение:** `Charged Off` — прокси списания, не доказательство просрочки 90+ дней. Статусы завершённых кредитов могут приводить к смещению выборки. Все расчёты прибыли условные. Не используйте для реальных кредитных решений.

Ноутбуки запускаются последовательно 01 → 02 → 03 → 04. Каталог `artifacts/` сохраняет промежуточные результаты. Один раз разместите архив `accepted_2007_to_2018Q4.csv.gz` в текущем каталоге Colab или укажите путь в ячейке ниже. При новом сеансе Colab локальные файлы пропадают: используйте Google Drive или загрузите готовые промежуточные файлы.

Зависимости: `pip install pandas numpy matplotlib seaborn scikit-learn lightgbm optbinning shap joblib pyyaml`.

## 04. Оценка стратегии, SHAP, PSI (test только для итоговой проверки)

In [ ]:
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
os.makedirs('artifacts', exist_ok=True)
def create_features(data):
    d=data.copy()
    num=['annual_inc','loan_amnt','installment','dti','fico_range_low','fico_range_high','revol_bal','open_acc','total_acc','delinq_2yrs']
    for c in num: d[c]=pd.to_numeric(d[c], errors='coerce')
    d['income_monthly']=d['annual_inc']/12
    d['loan_to_income']=d['loan_amnt']/(d['annual_inc']+1)
    d['payment_to_income']=d['installment']/(d['income_monthly']+1)
    d['debt_to_income']=d['dti']/100
    d['fico_average']=(d['fico_range_low']+d['fico_range_high'])/2
    d['fico_spread']=d['fico_range_high']-d['fico_range_low']
    d['credit_per_account']=d['revol_bal']/(d['open_acc']+1)
    d['account_activity']=d['open_acc']/(d['total_acc']+1)
    d['delinq_per_account']=d['delinq_2yrs']/(d['total_acc']+1)
    d['revolving_utilization']=pd.to_numeric(d['revol_util'].astype(str).str.replace('%','',regex=False),errors='coerce')/100
    return d
from sklearn.metrics import roc_auc_score, roc_curve
def metrics(y,p):
    auc=roc_auc_score(y,p);fpr,tpr,_=roc_curve(y,p)
    return {'ROC-AUC':auc,'Gini':2*auc-1,'KS':float(np.max(tpr-fpr))}
def profit(y,p,loan,threshold,good_return=.15,lgd=.45):
    y=np.asarray(y); p=np.asarray(p); loan=np.asarray(loan)
    return float(np.where(y==0,good_return*loan,-lgd*loan)[p<threshold].sum())

import shap
bundle=joblib.load('artifacts/model_bundle.pkl')
model=bundle['model'];binning=bundle['binning'];f=bundle['features']
valid=create_features(pd.read_pickle('artifacts/valid.pkl'))
test=create_features(pd.read_pickle('artifacts/test.pkl'))
Xv=binning.transform(valid[f].replace([np.inf,-np.inf],np.nan),metric='woe')
Xt=binning.transform(test[f].replace([np.inf,-np.inf],np.nan),metric='woe')
p_valid=model.predict_proba(Xv)[:,1]
# Порог выбираем по validation, тест не используется для подбора.
thresholds=np.arange(.05,.96,.05)
profits=[profit(valid.target,p_valid,valid.loan_amnt,t) for t in thresholds]
threshold=float(thresholds[int(np.argmax(profits))])
print('Порог (validation):',threshold)
p_test=model.predict_proba(Xt)[:,1]
print('TEST:',metrics(test.target,p_test))
print('Иллюстративная прибыль TEST:',profit(test.target,p_test,test.loan_amnt,threshold),'USD')
print('Ожидаемые потери без калибровки PD — только демонстрация:',np.sum(p_test*.45*test.loan_amnt.to_numpy()))


In [ ]:
sample=Xt.sample(n=min(500,len(Xt)),random_state=42)
sv=shap.TreeExplainer(model)(sample)
shap.summary_plot(sv.values,sample,max_display=10)
for i in range(min(3,len(sample))):
    shap.plots.waterfall(sv[i],max_display=10)


In [ ]:
def psi(expected,actual,bins=10):
    expected=np.asarray(expected,dtype=float);actual=np.asarray(actual,dtype=float)
    expected=expected[np.isfinite(expected)];actual=actual[np.isfinite(actual)]
    if len(expected)<2 or len(actual)<2:return np.nan
    cuts=np.unique(np.quantile(expected,np.linspace(0,1,bins+1)))
    if len(cuts)<2:return 0.0
    cuts[0],cuts[-1]=-np.inf,np.inf
    pe=np.clip(np.histogram(expected,cuts)[0]/len(expected),1e-6,None)
    pa=np.clip(np.histogram(actual,cuts)[0]/len(actual),1e-6,None)
    return float(np.sum((pa-pe)*np.log(pa/pe)))
train=create_features(pd.read_pickle('artifacts/train.pkl'))
Xtrain=binning.transform(train[f].replace([np.inf,-np.inf],np.nan),metric='woe')
p_train=model.predict_proba(Xtrain)[:,1]
print('PSI скора:',psi(p_train,p_test))
top5=pd.Series(model.feature_importances_,index=f).nlargest(5).index
for c in top5:print(c,'PSI исходного признака:',psi(train[c],test[c]))
joblib.dump({**bundle,'threshold':threshold},'model.pkl')
print('model.pkl сохранён; это учебная, некалиброванная модель')


**Отчёт:** описать выборку, источники, прокси-цель, экономику 10 признаков, сравнение ROC/Gini/KS, иллюстративные прибыль и EL, SHAP трёх наблюдений, PSI скора и топ-5 признаков, ограничения и отсутствие полноценной калибровки. Результаты исходного полного Colab могут отличаться из-за нового набора признаков и очистки.